# Анализ качества данных и очистка

Ноутбук предназначен для исследования таблицы `flats_features`, созданной DAG `build_flats_features`. Перед запуском выгрузите данные из PostgreSQL в CSV либо укажите путь к локальному CSV.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "part1_airflow").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "part1_airflow").is_dir():
    raise FileNotFoundError("Не найден корень mle-project-sprint-1-v001")
PLUGIN_DIR = PROJECT_ROOT / "part1_airflow" / "plugins"
sys.path.insert(0, str(PLUGIN_DIR.resolve()))
from cleaning import diagnose_data, clean_flats


Функции вынесены в `part1_airflow/plugins/cleaning.py`, чтобы один и тот же проверенный код использовался и в анализе, и в Airflow DAG. Следующая ячейка выводит определения функций в ноутбук.


In [2]:
from inspect import getsource

print("Реализация diagnose_data:")
print(getsource(diagnose_data))
print("Реализация clean_flats:")
print(getsource(clean_flats))


Реализация diagnose_data:
def diagnose_data(data: pd.DataFrame) -> dict:
    """Return a compact data-quality report for a notebook or a script."""
    report = {
        "rows": int(len(data)),
        "columns": int(data.shape[1]),
        "duplicate_rows": int(data.duplicated().sum()),
        "missing_by_column": data.isna().sum().astype(int).to_dict(),
    }
    if "flat_id" in data.columns:
        report["duplicate_flat_ids"] = int(data["flat_id"].duplicated().sum())
    elif "id" in data.columns:
        report["duplicate_flat_ids"] = int(data["id"].duplicated().sum())
    return report

Реализация clean_flats:
def clean_flats(data: pd.DataFrame) -> pd.DataFrame:
    """Clean the joined flats dataset while retaining model-ready columns.

    Rows with a missing/non-positive target or non-positive total area are
    removed. Implausible floor/area combinations are rejected. Extreme
    price-per-square-metre values are filtered using the 1st/99th percentiles.
    Remaining predi

In [3]:
DATA_PATH = PROJECT_ROOT / "part2_dvc" / "data" / "raw" / "flats_features.csv"
df = pd.read_csv(DATA_PATH)
df.head()


,flat_id,building_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,price_per_sqm
0,0,6220,1965,6,55.717113,37.781120,2.64,84,12,True,9,9.9,19.900000,1,False,False,35.099998,9500000,270655.282421
1,1,18012,2001,2,55.794849,37.608013,3.00,97,10,True,7,0.0,16.600000,1,False,False,43.000000,13500000,313953.488372
2,2,17821,2000,4,55.740040,37.761742,2.70,80,10,True,9,9.0,32.000000,2,False,False,56.000000,13500000,241071.428571
3,3,18579,2002,4,55.672016,37.570877,2.64,771,17,True,1,10.1,43.099998,3,False,False,76.000000,20000000,263157.894737
4,4,9293,1971,1,55.808807,37.707306,2.60,208,9,True,3,3.0,14.000000,1,False,False,24.000000,5200000,216666.666667


In [4]:
print("Размер исходных данных:", df.shape)
report = diagnose_data(df)
report


Размер исходных данных: (141362, 19)


{'rows': 141362,
 'columns': 19,
 'duplicate_rows': 0,
 'missing_by_column': {'flat_id': 0,
  'building_id': 0,
  'build_year': 0,
  'building_type_int': 0,
  'latitude': 0,
  'longitude': 0,
  'ceiling_height': 0,
  'flats_count': 0,
  'floors_total': 0,
  'has_elevator': 0,
  'floor': 0,
  'kitchen_area': 0,
  'living_area': 0,
  'rooms': 0,
  'is_apartment': 0,
  'studio': 0,
  'total_area': 0,
  'price': 0,
  'price_per_sqm': 0},
 'duplicate_flat_ids': 0}

In [5]:
missing_share = (df.isna().mean() * 100).sort_values(ascending=False)
missing_share.to_frame("missing_percent")


,missing_percent
flat_id,0.0
building_id,0.0
build_year,0.0
building_type_int,0.0
latitude,0.0
longitude,0.0
ceiling_height,0.0
flats_count,0.0
floors_total,0.0
has_elevator,0.0


In [6]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
df[numeric_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
flat_id,141362.0,7.068050e+04,4.080784e+04,0.000000,3.534025e+04,7.068050e+04,1.060208e+05,1.413610e+05
building_id,141362.0,1.405367e+04,6.988831e+03,1.000000,8.535250e+03,1.433200e+04,2.047500e+04,2.462000e+04
build_year,141362.0,1.986600e+03,2.213641e+01,1901.000000,1.969000e+03,1.985000e+03,2.007000e+03,2.023000e+03
building_type_int,141362.0,3.232941e+00,1.459461e+00,0.000000,2.000000e+00,4.000000e+00,4.000000e+00,6.000000e+00
latitude,141362.0,5.573006e+01,1.026111e-01,55.211460,5.565386e+01,5.572469e+01,5.580732e+01,5.601103e+01
longitude,141362.0,3.758923e+01,1.501218e-01,36.864372,3.749176e+01,3.758115e+01,3.769106e+01,3.794641e+01
ceiling_height,141362.0,2.753650e+00,2.232754e-01,2.000000,2.640000e+00,2.640000e+00,2.800000e+00,2.700000e+01
flats_count,141362.0,2.519932e+02,2.073362e+02,1.000000,1.110000e+02,2.000000e+02,3.240000e+02,4.455000e+03
floors_total,141362.0,1.410755e+01,6.898045e+00,1.000000,9.000000e+00,1.400000e+01,1.700000e+01,9.900000e+01
floor,141362.0,7.467346e+00,5.717144e+00,1.000000,3.000000e+00,6.000000e+00,1.000000e+01,5.600000e+01


## Очистка

Функция удаляет дубликаты и записи с некорректной целевой ценой/площадью, проверяет этажность и выбросы цены за квадратный метр, затем заполняет пропуски в признаках медианой или модой. `price_per_sqm` используется только для контроля качества и не должен попадать в признаки модели.


In [7]:
cleaned = clean_flats(df)
print("До:", df.shape, "После:", cleaned.shape)
cleaned.head()


До: (141362, 19) После: (138534, 19)


,flat_id,building_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,price_per_sqm
0,0,6220,1965,6,55.717113,37.781120,2.64,84,12,True,9,9.9,19.900000,1,False,False,35.099998,9500000,270655.282421
1,1,18012,2001,2,55.794849,37.608013,3.00,97,10,True,7,0.0,16.600000,1,False,False,43.000000,13500000,313953.488372
2,2,17821,2000,4,55.740040,37.761742,2.70,80,10,True,9,9.0,32.000000,2,False,False,56.000000,13500000,241071.428571
3,3,18579,2002,4,55.672016,37.570877,2.64,771,17,True,1,10.1,43.099998,3,False,False,76.000000,20000000,263157.894737
4,4,9293,1971,1,55.808807,37.707306,2.60,208,9,True,3,3.0,14.000000,1,False,False,24.000000,5200000,216666.666667


In [8]:
diagnose_data(cleaned)


{'rows': 138534,
 'columns': 19,
 'duplicate_rows': 0,
 'missing_by_column': {'flat_id': 0,
  'building_id': 0,
  'build_year': 0,
  'building_type_int': 0,
  'latitude': 0,
  'longitude': 0,
  'ceiling_height': 0,
  'flats_count': 0,
  'floors_total': 0,
  'has_elevator': 0,
  'floor': 0,
  'kitchen_area': 0,
  'living_area': 0,
  'rooms': 0,
  'is_apartment': 0,
  'studio': 0,
  'total_area': 0,
  'price': 0,
  'price_per_sqm': 0},
 'duplicate_flat_ids': 0}

In [9]:
cleaned.to_csv("flats_cleaned_preview.csv", index=False)
print("Сохранён flats_cleaned_preview.csv")


Сохранён flats_cleaned_preview.csv
